In [5]:
import pandas as pd

def mapear_nomes_colunas(df):
    """
    Identifica as colunas originais por palavras-chave e aplica os nomes personalizados.
    """
    mapeamento = {}
    for col in df.columns:
        col_lower = col.lower()

        # Ignora o carimbo, pois ele será excluído depois
        if 'carimbo' in col_lower:
            continue
        elif 'gripado' in col_lower:
            mapeamento[col] = 'ficou_gripado'
        elif 'vacina' in col_lower:
            mapeamento[col] = 'tomou_vacina'
        elif 'frequentou' in col_lower:
            mapeamento[col] = 'frequentou_ambiente_multidao'
        elif 'viajou' in col_lower:
            mapeamento[col] = 'você_viajou'
        elif 'alergia' in col_lower:
            mapeamento[col] = 'tem_alergia'
        elif 'dormiu' in col_lower:
            mapeamento[col] = 'horas_dormidas'
        elif 'atividade' in col_lower or 'praticou' in col_lower:
            mapeamento[col] = 'praticou_exercicios'
        elif 'alimentou' in col_lower or 'balanceada' in col_lower:
            mapeamento[col] = 'alimentação_saudavel'
        elif 'lavou' in col_lower or 'vezes' in col_lower:
            mapeamento[col] = 'lavou_maos'
        elif 'estresse' in col_lower:
            mapeamento[col] = 'nivel_estresse'
        else:
            # Fallback caso exista alguma coluna nova não mapeada
            mapeamento[col] = 'coluna_desconhecida'

    return mapeamento

def carregar_e_preparar_dados(url):
    """
    Carrega do Google Sheets, remove a coluna de Data/Hora e aplica o mapeamento de colunas.
    """
    print("[1/3] Carregando e processando dados da planilha...")
    sheet_id = url.split('/d/')[1].split('/')[0]
    gid = url.split('gid=')[1]
    csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"

    df = pd.read_csv(csv_url)

    # 1. Ignorar a coluna de 'Carimbo de data/hora'
    if 'Carimbo de data/hora' in df.columns:
        df = df.drop(columns=['Carimbo de data/hora'])
        print("      - Coluna 'Carimbo de data/hora' removida.")

    # 2. Aplicar o mapeamento exato solicitado
    dicionario_novas_colunas = mapear_nomes_colunas(df)
    df = df.rename(columns=dicionario_novas_colunas)

    print("      - Colunas renomeadas conforme o mapeamento personalizado:")
    for col_antiga, col_nova in dicionario_novas_colunas.items():
        print(f"        * '{col_antiga[:35]}...' -> '{col_nova}'")

    print(f"\n      Tamanho final do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas.\n")
    return df

In [6]:
def treinar_prism(df, coluna_alvo):
    """
    Implementação do algoritmo PRISM do zero.
    """
    print(f"[2/3] Iniciando o treinamento do algoritmo PRISM (Alvo: '{coluna_alvo}')...")
    regras = []

    # Remove possíveis valores nulos da coluna alvo
    df = df.dropna(subset=[coluna_alvo])
    classes_alvo = df[coluna_alvo].unique()

    for classe in classes_alvo:
        print(f"\n{'='*50}")
        print(f"=== Extraindo regras para a classe: {classe} ===")
        print(f"{'='*50}")

        df_classe = df.copy()

        while len(df_classe[df_classe[coluna_alvo] == classe]) > 0:
            regra_atual = {}
            subconjunto = df_classe.copy()

            print("\n  -> Buscando nova regra...")

            while len(subconjunto[subconjunto[coluna_alvo] != classe]) > 0:
                melhor_condicao = None
                melhor_precisao = -1
                melhor_cobertura = -1

                for coluna in df.columns:
                    if coluna == coluna_alvo or coluna in regra_atual:
                        continue

                    valores_unicos = subconjunto[coluna].dropna().unique()

                    for valor in valores_unicos:
                        match_condicao = subconjunto[subconjunto[coluna] == valor]
                        total_match = len(match_condicao)

                        if total_match == 0:
                            continue

                        match_classe = len(match_condicao[match_condicao[coluna_alvo] == classe])
                        precisao = match_classe / total_match

                        # Seleciona a condição com maior precisão e, em caso de empate, maior cobertura
                        if precisao > melhor_precisao or (precisao == melhor_precisao and match_classe > melhor_cobertura):
                            melhor_precisao = precisao
                            melhor_cobertura = match_classe
                            melhor_condicao = (coluna, valor)

                if melhor_condicao is None:
                    print("     [Aviso] Dados inconsistentes encontrados. Interrompendo regra.")
                    break

                atributo, valor = melhor_condicao
                regra_atual[atributo] = valor
                subconjunto = subconjunto[subconjunto[atributo] == valor]

                print(f"     + Condição: {atributo} = '{valor}' | (Precisão: {melhor_precisao*100:.1f}%, Cobertura: {melhor_cobertura})")

            regras.append({
                'condicoes': regra_atual,
                'classe': classe
            })
            print(f"  [!] Regra finalizada: SE {regra_atual} ENTÃO {coluna_alvo} = '{classe}'")

            mascara_cobertura = pd.Series([True] * len(df_classe), index=df_classe.index)
            for atributo, valor in regra_atual.items():
                mascara_cobertura = mascara_cobertura & (df_classe[atributo] == valor)

            instancias_cobertas = df_classe[mascara_cobertura & (df_classe[coluna_alvo] == classe)]
            df_classe = df_classe.drop(instancias_cobertas.index)
            print(f"  [-] {len(instancias_cobertas)} instâncias da classe '{classe}' removidas.")

    return regras

In [7]:
def exibir_regras_legiveis(regras, coluna_alvo):
    """
    Apresenta as regras de forma limpa no console usando as nomenclaturas customizadas.
    """
    print("\n[3/3] Resultados Finais\n")
    print("="*60)
    print(" LISTA CLASSIFICADA DE REGRAS (MODELO PRISM)")
    print("="*60)

    for i, regra in enumerate(regras, 1):
        lista_condicoes = [f"[{k} é igual a '{v}']" for k, v in regra['condicoes'].items()]
        str_condicoes = " E\n        ".join(lista_condicoes) if lista_condicoes else "TODOS OS CASOS"

        print(f"Regra {i:02d}:")
        print(f"  SE    {str_condicoes}")
        print(f"  ENTÃO -> {coluna_alvo} é '{regra['classe']}'")
        print("-" * 60)

In [8]:
# ==========================================
# EXECUÇÃO DO FLUXO
# ==========================================
if __name__ == "__main__":
    url_planilha = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/edit?gid=282951434#gid=282951434"

    # 1. Carregar os dados (já com o pré-processamento de nomes e drop do carimbo)
    df_dados = carregar_e_preparar_dados(url_planilha)

    # A coluna alvo agora será 'ficou_gripado' por conta do nosso processamento
    NOME_DA_COLUNA_ALVO = 'ficou_gripado'

    if NOME_DA_COLUNA_ALVO in df_dados.columns:
        # 2. Treinar o modelo e gerar as regras
        regras_extraidas = treinar_prism(df_dados, NOME_DA_COLUNA_ALVO)

        # 3. Exibir as regras legíveis
        exibir_regras_legiveis(regras_extraidas, NOME_DA_COLUNA_ALVO)
    else:
        print(f"Erro: A coluna alvo '{NOME_DA_COLUNA_ALVO}' não foi encontrada após o pré-processamento.")
        print(f"Colunas disponíveis: {df_dados.columns.tolist()}")

[1/3] Carregando e processando dados da planilha...
      - Coluna 'Carimbo de data/hora' removida.
      - Colunas renomeadas conforme o mapeamento personalizado:
        * 'Você ficou gripado no ano passado ?...' -> 'ficou_gripado'
        * 'Você tomou vacina da gripe no ano p...' -> 'tomou_vacina'
        * '  Você frequentou no ano passado,  ...' -> 'frequentou_ambiente_multidao'
        * '  Você viajou no ano passado mais d...' -> 'você_viajou'
        * '  Você tem alergia nas vias aéreas ...' -> 'tem_alergia'
        * 'Quantas horas você dormiu em média ...' -> 'horas_dormidas'
        * 'Você praticou atividade física no a...' -> 'praticou_exercicios'
        * 'Você se alimentou de forma balancea...' -> 'alimentação_saudavel'
        * 'Em média, quantas vezes você lavou ...' -> 'lavou_maos'
        * 'Na sua percepção, o seu nível de es...' -> 'nivel_estresse'

      Tamanho final do dataset: 186 linhas e 10 colunas.

[2/3] Iniciando o treinamento do algoritmo PRISM (Alvo: